### Visão Geral
Script responsável por orquestrar a extração de dados de múltiplos recursos de uma API pública (Atas, Notas de Empenho, Liquidações, Contratos, Contratações e Servidores), iterando por todos os anos de um intervalo configurável, salvando os resultados em arquivos CSV e gerando um relatório consolidado ao final da execução.

```sh
┌─────────────────────────────────────┐
│  Para cada recurso (6 extratores)   │
│  ┌───────────────────────────────┐  │
│  │  Para cada ano (2025 → atual) │  │
│  │  ┌─────────────────────────┐  │  │
│  │  │ Instanciar extrator     │  │  │
│  │  │ extract_and_save_year() │  │  │
│  │  │ try/except → registro   │  │  │
│  │  │ notificar_extracao()    │  │  │
│  │  │ sleep(1s)               │  │  │
│  │  └─────────────────────────┘  │  │
│  └───────────────────────────────┘  │
└─────────────────────────────────────┘
              ↓
     Salvar relatorio_extracao.json
              ↓
     Exibir resumo consolidado
```



In [ ]:
# Import necessary libraries
import sys
from src.api import DadosAbertosPBAPI
from typing import Any, Dict, List
from pathlib import Path
import os
import subprocess
from datetime import datetime
import json
import traceback
import time
import duckdb
from pathlib import Path

api = DadosAbertosPBAPI()

### Requisição aos dados

A execução extrai diretamente da api de acesso aberto da SEAD-PB. Pode demorar certo tempo, não é uniforme, e considera-se que os dados requisitados não sejam recentes.



In [ ]:
from IPython.display import display, Markdown

from src.extractors import (
    AtasExtractor,
    NotasEmpenhoExtractor,
    LiquidacoesExtractor,
    ContratosExtractor,
    ContratacoesExtractor,
    ServidoresExtractor,
)

ANO_INICIAL = 2025
ANO_FINAL = datetime.now().year

ANOS = range(ANO_INICIAL, ANO_FINAL + 1)

FORMATOS = ["csv"]
OUTPUT_DIR = Path("./data/output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DIRETORIO_DADOS = str(OUTPUT_DIR)

extratores = {
    "atas": AtasExtractor,
    "notas_empenho": NotasEmpenhoExtractor,
    "liquidacoes": LiquidacoesExtractor,
    "contratos": ContratosExtractor,
    "contratacoes": ContratacoesExtractor,
    "servidores": ServidoresExtractor,
}

relatorio_geral = []


def notificar_extracao(nome: str, ano: int, registro: Dict[str, Any]) -> None:
    sucesso = registro["status"] == "sucesso"
    titulo = "Extração concluída" if sucesso else "Falha na extração"
    mensagem = (
        f"{nome} ({ano}): {registro['total_registros']} registros"
        if sucesso
        else f"{nome} ({ano}): {registro['erro']}"
    )

    try:
        resultado_notificacao = subprocess.run(
            ["notify-send", "--app-name=Lover Letter Lib", titulo, mensagem],
            capture_output=True,
            text=True,
            check=False,
        )
    except OSError as erro:
        display(Markdown(f"**Não foi possível enviar a notificação:** `{erro}`"))
        return

    if resultado_notificacao.returncode != 0:
        detalhe = resultado_notificacao.stderr.strip()
        display(Markdown(f"**Falha ao enviar a notificação:** `{detalhe}`"))


for nome, classe_extrator in extratores.items():

    display(Markdown(f"---\n## RECURSO: `{nome}`"))

    for ano in ANOS:

        display(Markdown(f"**Extraindo `{nome}` de `{ano}`...**"))

        try:
            extrator = classe_extrator(
                api=api,
                output_dir=DIRETORIO_DADOS,
            )

            resultado = extrator.extract_and_save_year(
                ano,
                formats=FORMATOS,
                include_report=True,
            )

            registro = {
                "recurso": nome,
                "ano": ano,
                "status": "sucesso",
                "total_registros": resultado.get("total_registros", 0),
                "arquivos": resultado.get("arquivos", []),
            }

            arquivos_md = "\n".join(f"  - `{a}`" for a in registro["arquivos"]) or "  - _(nenhum)_"
            display(Markdown(
                f"**Sucesso** — `{registro['total_registros']}` registros\n\n"
                f"**Arquivos:**\n{arquivos_md}"
            ))

        except Exception as erro:

            registro = {
                "recurso": nome,
                "ano": ano,
                "status": "erro",
                "erro": str(erro),
            }

            display(Markdown(
                f"**ERRO em `{nome}`, ano `{ano}`:** `{erro}`"
            ))
            traceback.print_exc()

        relatorio_geral.append(registro)
        notificar_extracao(nome, ano, registro)

        time.sleep(1)


caminho_relatorio = OUTPUT_DIR / "relatorio_extracao.json"

with open(caminho_relatorio, "w", encoding="utf-8") as arquivo:
    json.dump(
        relatorio_geral,
        arquivo,
        ensure_ascii=False,
        indent=2,
        default=str,
    )


sucessos = sum(
    item["status"] == "sucesso"
    for item in relatorio_geral
)

erros = sum(
    item["status"] == "erro"
    for item in relatorio_geral
)

total_registros = sum(
    item.get("total_registros", 0)
    for item in relatorio_geral
)

display(Markdown(
    "---\n"
    "#  RESUMO DA EXTRAÇÃO\n\n"
    f"- **Execuções bem-sucedidas:** `{sucessos}`\n"
    f"- **Execuções com erro:** `{erros}`\n"
    f"- **Total de registros reportados:** `{total_registros}`\n"
    f"- **Relatório:** `{caminho_relatorio}`\n"
    f"- **Dados armazenados em:** `{OUTPUT_DIR}`"
))

### Objetivo

carrega arquivos CSV (extraídos previamente para data/output/) em um banco DuckDB (sead.duckdb), criando 6 tabelas com tipagem explícita — garantindo consistência de schema e permitindo consultas SQL rápidas sobre os dados.


| Tabela | Origem | Colunas-chave | Tipos aplicados |
|---|---|---|---|
| `atas` | `atas_2025.csv` | `numeroProcesso`, `ano`, `mes`, `nomeOrgao`, `modalidade`, `situacao`, `valorLicitado` | `VARCHAR`, `INTEGER`, `DOUBLE` |
| `contratacoes` | `contratacoes_2025.csv` | `numeroProcesso`, `numeroLicitacao`, `cadastroCge`, `valorEstimado`, `valorAdjudicado` | `VARCHAR`, `DOUBLE` |
| `contratos` | `contratos_2025.csv` | `numeroContrato`, `registroCge`, `cnpjCpf`, `cpfGestor`, `valorTotal`, vigências | `VARCHAR`, `DOUBLE` |
| `notas_empenho` | `notas_empenho_2025.csv` | `numeroEmpenho`, `codigoOrgao`, `contrato`, `valorEmpenhado`, `valorLiquidado`, `valorPago` | `BIGINT`, `INTEGER`, `DOUBLE` |
| `liquidacoes` | `liquidacoes_2025.csv` | `numeroEmpenho`, `cnpjCpfCredor`, `nomeCredor`, `valorEmpenhado`, `valorGD` | `BIGINT`, `INTEGER`, `DOUBLE` |
| `servidores` | `servidores_2025.csv` | `cpfServidor`, `matricula`, `nomeServidor`, `nomeCargo`, `valorBruto`, `valorLiquido` | `INTEGER`, `DOUBLE`, `VARCHAR` |


In [ ]:
DB_PATH = "sead.duckdb"
DATA_DIR = Path("data/output")
YEAR = "2025"

CSV_OPTS = "header=true, nullstr=['', 'NULL', 'null', 'NA', 'N/A']"


def load_table(con: duckdb.DuckDBPyConnection,
             tabela: str,
             csv_path: Path,
             select_sql: str) -> None:
    """Cria tabela com tipos explícitos e carrega do CSV."""
    if not csv_path.exists():
        display(Markdown(f"[SKIP] {csv_path} não encontrado"))
        return
    con.execute(f"DROP TABLE IF EXISTS {tabela}")
    con.execute(f"""
        CREATE TABLE {tabela} AS
        {select_sql}
        FROM read_csv_auto('{csv_path.as_posix()}', {CSV_OPTS})
    """)
    n = con.execute(f"SELECT COUNT(*) FROM {tabela}").fetchone()[0]
    display(Markdown(f"[OK] {tabela:<15} {n:>10,} linhas"))

con = duckdb.connect(DB_PATH)

load_table(con, "atas",
    DATA_DIR  / f"atas_{YEAR}.csv",
    """
    SELECT
        CAST(numeroProcesso AS VARCHAR)   AS numeroProcesso,
        TRY_CAST(ano AS INTEGER)          AS ano,
        TRY_CAST(mes AS INTEGER)          AS mes,
        CAST(nomeOrgao AS VARCHAR)        AS nomeOrgao,
        CAST(tipoProcesso AS VARCHAR)     AS tipoProcesso,
        CAST(modalidade AS VARCHAR)       AS modalidade,
        CAST(situacao AS VARCHAR)         AS situacao,
        TRY_CAST(valorLicitado AS DOUBLE) AS valorLicitado
    """)

load_table(con, "contratacoes",
    DATA_DIR  / f"contratacoes_{YEAR}.csv",
    """
    SELECT
        CAST(numeroProcesso AS VARCHAR)    AS numeroProcesso,
        CAST(numeroLicitacao AS VARCHAR)   AS numeroLicitacao,
        CAST(cadastroCge AS VARCHAR)       AS cadastroCge,
        CAST(nomeOrgao AS VARCHAR)         AS nomeOrgao,
        CAST(modalidade AS VARCHAR)        AS modalidade,
        CAST(situacao AS VARCHAR)          AS situacao,
        TRY_CAST(valorEstimado AS DOUBLE)  AS valorEstimado,
        TRY_CAST(valorAdjudicado AS DOUBLE) AS valorAdjudicado
    """)

load_table(con, "contratos",
    DATA_DIR  / f"contratos_{YEAR}.csv",
    """
    SELECT
        CAST(numeroContrato AS VARCHAR)             AS numeroContrato,
        NULLIF(TRIM(CAST(numeroProcessoLicitacao AS VARCHAR)), '')
                                                    AS numeroProcessoLicitacao,
        CAST(registroCge AS VARCHAR)                AS registroCge,
        CAST(codigoOrgao AS VARCHAR)                AS codigoOrgao,
        NULLIF(TRIM(CAST(cnpjCpf AS VARCHAR)), '')  AS cnpjCpf,
        NULLIF(TRIM(CAST(cpfGestor AS VARCHAR)), '') AS cpfGestor,
        NULLIF(TRIM(CAST(matriculaGestor AS VARCHAR)), '')
                                                    AS matriculaGestor,
        CAST(nomeOrgao AS VARCHAR)                  AS nomeOrgao,
        CAST(nomeGestor AS VARCHAR)                 AS nomeGestor,
        TRY_CAST(valorTotal AS DOUBLE)              AS valorTotal,
        CAST(dataInicioVigencia  AS VARCHAR)        AS dataInicioVigencia,
        CAST(dataTerminoVigencia AS VARCHAR)        AS dataTerminoVigencia
    """)

load_table(con, "notas_empenho",
    DATA_DIR  / f"notas_empenho_{YEAR}.csv",
    """
    SELECT
        TRY_CAST(numeroEmpenho AS BIGINT)               AS numeroEmpenho,
        TRY_CAST(codigoOrgao  AS BIGINT)                AS codigoOrgao,
        TRY_CAST(ano          AS INTEGER)               AS ano,
        NULLIF(TRIM(CAST(contrato AS VARCHAR)), '')     AS contrato,
        NULLIF(TRIM(CAST(processo AS VARCHAR)), '')     AS processo,
        NULLIF(TRIM(CAST(numeroProcessoCompras AS VARCHAR)), '')
                                                        AS numeroProcessoCompras,
        TRY_CAST(codigoLicitacao AS DOUBLE)             AS codigoLicitacao,
        NULLIF(TRIM(CAST(registroCGE AS VARCHAR)), '')  AS registroCGE,
        NULLIF(TRIM(CAST(cpfCnpj AS VARCHAR)), '')      AS cpfCnpj,
        CAST(nomeOrgao AS VARCHAR)                      AS nomeOrgao,
        TRY_CAST(valorEmpenhado AS DOUBLE)              AS valorEmpenhado,
        TRY_CAST(valorLiquidado AS DOUBLE)              AS valorLiquidado,
        TRY_CAST(valorPago      AS DOUBLE)              AS valorPago
    """)

load_table(con, "liquidacoes",
    DATA_DIR  / f"liquidacoes_{YEAR}.csv",
    """
    SELECT
        TRY_CAST(numeroEmpenho AS BIGINT)                AS numeroEmpenho,
        TRY_CAST(codigoOrgao  AS BIGINT)                 AS codigoOrgao,
        TRY_CAST(anoExercicio AS INTEGER)                AS anoExercicio,
        NULLIF(TRIM(CAST(cnpjCpfCredor AS VARCHAR)), '') AS cnpjCpfCredor,
        TRY_CAST(numeroDocumento AS BIGINT)              AS numeroDocumento,
        CAST(nomeCredor AS VARCHAR)                      AS nomeCredor,
        TRY_CAST(valorEmpenhado AS DOUBLE)               AS valorEmpenhado,
        TRY_CAST(valorGD        AS DOUBLE)               AS valorGD
    """)

load_table(con, "servidores",
    DATA_DIR  / f"servidores_{YEAR}.csv",
    """
    SELECT
        NULLIF(TRIM(CAST(cpfServidor AS VARCHAR)), '')   AS cpfServidor,
        NULLIF(TRIM(CAST(matricula   AS VARCHAR)), '')   AS matricula,
        TRY_CAST(anoExercicio  AS INTEGER)               AS anoExercicio,
        TRY_CAST(mesReferencia AS INTEGER)               AS mesReferencia,
        TRY_CAST(periodo       AS INTEGER)               AS periodo,
        CAST(nomeServidor AS VARCHAR)                    AS nomeServidor,
        CAST(orgaoLotacao AS VARCHAR)                    AS orgaoLotacao,
        NULLIF(TRIM(CAST(cnpjOrgao AS VARCHAR)), '')     AS cnpjOrgao,
        CAST(nomeUnidadeTrabalho AS VARCHAR)             AS nomeUnidadeTrabalho,
        CAST(siglaPoder AS VARCHAR)                      AS siglaPoder,
        CAST(nomeCargo  AS VARCHAR)                      AS nomeCargo,
        CAST(tipoCargo  AS VARCHAR)                      AS tipoCargo,
        CAST(situacaoServidor AS VARCHAR)                AS situacaoServidor,
        CAST(dataAdmissao AS VARCHAR)                    AS dataAdmissao,
        TRY_CAST(valorBruto  AS DOUBLE)                  AS valorBruto,
        TRY_CAST(valorLiquido AS DOUBLE)                 AS valorLiquido
    """)

con.close()

In [ ]:
import duckdb
con = duckdb.connect("sead.duckdb")
con.execute(open("./lib/sead_indexes.sql").read())
con.execute(open("./lib/validation.sql").read())
con.execute(open("./lib/main_chain.sql").read())

con.close()

In [ ]:

from IPython.display import display

df = con.execute("SELECT * FROM v_cadeia_principal LIMIT 10").df()

df2 = con.execute("""
    SELECT *
    FROM notas_empenho n
    JOIN liquidacoes l USING (numeroEmpenho, codigoOrgao)
    WHERE n.ano = l.anoExercicio
    LIMIT 10
""").df()

display(df2)
# Alertas do diagrama
print(con.execute("SELECT COUNT(*) FROM v_dup_atas").fetchone()[0], "processos duplicados em Atas")
print(con.execute("SELECT COUNT(*) FROM v_contratos_sem_processo").fetchone()[0], "contratos sem processo (esperado ~2.484)")

In [ ]:
import pandas as pd
atas = pd.read_csv("data/output/atas/atas_2025.csv")
contratos = pd.read_csv("data/output/contratos/contratos_2025.csv")
contratacoes = pd.read_csv("data/output/contratacoes/contratacoes_2025.csv")
liquidacoes = pd.read_csv("data/output/liquidacoes/liquidacoes_2025.csv")
notas_empenho = pd.read_csv("data/output/notas_empenho/notas_empenho_2025.csv")
servidores = pd.read_csv("data/output/servidores/servidores_2025.csv")
